# ARCADE stenosis segmentation: Kaggle GPU sweep

This notebook runs the existing project's training and validation code on Kaggle GPU. It discovers ARCADE input locations, reproduces the active baseline, trains controlled alternatives, and exports uniquely named checkpoints plus validation reports. Model selection and thresholds use train/validation only. The notebook does not open the test split.

SMP architecture/encoder API references: [model architectures](https://github.com/qubvel-org/segmentation_models.pytorch/blob/main/docs/models.rst) and [pretrained encoders](https://github.com/qubvel-org/segmentation_models.pytorch/blob/main/docs/encoders.rst).

In [ ]:
%pip -q install 'segmentation-models-pytorch==0.6.0' 'pydantic-settings>=2,<3' 

## Prepare private Kaggle inputs from Windows

1. From the project root, run `python .\notebooks\build_kaggle_bundle.py`.
2. Upload `kaggle_project_code.zip` as a private Kaggle Dataset. It includes only backend application code, the baseline checkpoint, its threshold sidecar, and optional ML requirements. It excludes `.env`, database, uploads, processed files, and ARCADE data.
3. Add this code dataset and the original ARCADE dataset with Kaggle **Add Input**. Select a GPU accelerator. Enable Internet to fetch ImageNet encoder weights; otherwise change `ENCODER_WEIGHTS` to `none` below.
4. Run the discovery cell. It prints verified input locations. If more than one ARCADE root is mounted, choose by displayed index.

In [ ]:
from pathlib import Path
import os, sys, zipfile, subprocess, json, time, shutil
INPUT_ROOT=Path('/kaggle/input')
assert INPUT_ROOT.is_dir(), 'Use Kaggle Add Input to mount the code and ARCADE datasets.'
def valid_root(p):
    needed=['stenosis/train/images','stenosis/val/images','stenosis/train/annotations/train.json','stenosis/val/annotations/val.json','syntax/train/annotations/train.json','syntax/val/annotations/val.json']
    return all((p/x).exists() for x in needed)
arcade_candidates=[]
for current, dirs, _ in os.walk(INPUT_ROOT):
    p=Path(current)
    if valid_root(p): arcade_candidates.append(p); dirs[:]=[]
print('ARCADE root candidates:')
for i,p in enumerate(arcade_candidates): print(f'[{i}] {p}')
assert arcade_candidates, 'No mounted ARCADE root matched the expected train/validation layout.'
ARCADE_ROOT=arcade_candidates[0] if len(arcade_candidates)==1 else arcade_candidates[int(input('Choose ARCADE root index: '))]
PROJECT_ROOT=Path('/kaggle/working/arcade_project_code')
PROJECT_ROOT.mkdir(parents=True,exist_ok=True)
archives=list(INPUT_ROOT.rglob('kaggle_project_code.zip'))
print('Code bundle candidates:',archives)
if len(archives)==1:
    with zipfile.ZipFile(archives[0]) as z: z.extractall(PROJECT_ROOT)
    BACKEND_ROOT=PROJECT_ROOT/'backend'
elif len(archives)==0:
    # Kaggle may expand ZIP files uploaded as dataset entries. Locate the
    # resulting backend source tree and copy it to writable working storage.
    experiment_files=list(INPUT_ROOT.rglob('train_experiment.py'))
    backend_candidates=[]
    for script in experiment_files:
        if script.parent.name!='arcade_ml' or script.parent.parent.name!='app': continue
        backend=script.parents[2]
        if (backend/'checkpoints/best_stenosis_unet_optimized.pth').is_file(): backend_candidates.append(backend)
    backend_candidates=list(dict.fromkeys(backend_candidates))
    print('Expanded backend candidates:',backend_candidates)
    assert len(backend_candidates)==1, 'Expected exactly one expanded backend with the baseline checkpoint.'
    shutil.copytree(backend_candidates[0],PROJECT_ROOT/'backend',dirs_exist_ok=True)
    BACKEND_ROOT=PROJECT_ROOT/'backend'
else:
    raise RuntimeError(f'Expected one code archive or expanded code tree, found {len(archives)} archives.')
assert (BACKEND_ROOT/'app/arcade_ml/train_experiment.py').is_file(), 'Project bundle is incomplete.'
assert (BACKEND_ROOT/'checkpoints/best_stenosis_unet_optimized.pth').is_file(), 'Baseline checkpoint is missing from the project bundle.'
os.environ['ARCADE_ROOT']=str(ARCADE_ROOT)
os.environ['COMPLETE_PROJECT_ROOT']=str(PROJECT_ROOT)
sys.path.insert(0,str(BACKEND_ROOT))
import torch
print('ARCADE_ROOT:',ARCADE_ROOT,'| torch:',torch.__version__,'| CUDA:',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Enable a Kaggle GPU accelerator before training.')
print('GPU:',torch.cuda.get_device_name(0),'| memory GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))

## Reproduce baseline before training

The evaluator uses all 200 validation images, app preprocessing, threshold selection by macro Dice, one-to-one candidate matching at IoU 0.30/0.50, and validation-only reports. Its default audit covers train/validation; the test split is not read.

In [ ]:
baseline=BACKEND_ROOT/'checkpoints/best_stenosis_unet_optimized.pth'
assert baseline.is_file()
env=os.environ.copy()
subprocess.run([sys.executable,'-u','-m','app.arcade_ml.audit_validation','--checkpoint',str(baseline),'--device','cuda'],cwd=BACKEND_ROOT,env=env,check=True)

## Experiment configurations

This sweep compares 256/384/512 input, a focal-Tversky ablation, pretrained ResNet34 and EfficientNet-B0 encoders, U-Net++, DeepLabV3+, and app-compatible Attention U-Net. Runs save separate checkpoints and reports. Resize batches downward if Kaggle GPU memory is insufficient. ImageNet normalization is used for SMP pretrained encoders in both training and app inference; scratch models use the baseline `/255` normalization.

In [ ]:
ENCODER_WEIGHTS='imagenet'  # set 'none' if Kaggle Internet is disabled
EPOCHS=16
PATIENCE=5
SEED=20261201
EXPERIMENTS=[
 dict(id='smp_unet_resnet34_256',architecture='smp_unet',encoder='resnet34',size=256,batch=8,loss='bce_dice',sample_weight=1.0),
 dict(id='smp_unet_resnet34_384',architecture='smp_unet',encoder='resnet34',size=384,batch=4,loss='bce_dice',sample_weight=1.0),
 dict(id='smp_unet_resnet34_512',architecture='smp_unet',encoder='resnet34',size=512,batch=2,loss='bce_dice',sample_weight=1.0),
 dict(id='smp_unet_resnet34_384_focal_tversky',architecture='smp_unet',encoder='resnet34',size=384,batch=4,loss='focal_tversky',sample_weight=1.0),
 dict(id='smp_unet_efficientnetb0_384',architecture='smp_unet',encoder='efficientnet-b0',size=384,batch=4,loss='bce_dice',sample_weight=1.0),
 dict(id='smp_unetplusplus_resnet34_384',architecture='smp_unetplusplus',encoder='resnet34',size=384,batch=2,loss='bce_dice',sample_weight=1.0),
 dict(id='smp_deeplabv3plus_resnet34_384',architecture='smp_deeplabv3plus',encoder='resnet34',size=384,batch=2,loss='bce_dice',sample_weight=1.0),
 dict(id='attention_unet_base16_384',architecture='attention_unet',encoder='none',size=384,batch=2,loss='bce_dice',sample_weight=1.5),
]
RUN_IDS={x['id'] for x in EXPERIMENTS}  # reduce this set for a shorter Kaggle session
CKPT_DIR=BACKEND_ROOT/'checkpoints'

def stream(cmd):
    p=subprocess.Popen(cmd,cwd=BACKEND_ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout: print(line,end='')
    return p.wait()

run_results=[]
for x in EXPERIMENTS:
    if x['id'] not in RUN_IDS: continue
    cp=CKPT_DIR/f"{x['id']}.pth"
    if not cp.exists():
        norm='imagenet' if x['architecture'].startswith('smp_') else 'none'
        command=[sys.executable,'-u','-m','app.arcade_ml.train_experiment','--architecture',x['architecture'],
          '--encoder-name',x['encoder'],'--encoder-weights',ENCODER_WEIGHTS if x['architecture'].startswith('smp_') else 'none',
          '--normalization',norm,'--loss',x['loss'],'--optimizer','adamw','--scheduler','cosine','--augmentation','conservative',
          '--positive-sample-weight',str(x['sample_weight']),'--epochs',str(EPOCHS),'--patience',str(PATIENCE),
          '--batch-size',str(x['batch']),'--grad-accumulation','2' if x['size']==512 else '1','--amp',
          '--image-size',str(x['size']),'--base-channels','16','--learning-rate','0.0003','--pos-weight','2',
          '--seed',str(SEED),'--output',str(cp)]
        code=stream(command)
        if code: print('Training exit code:',code,'; evaluating any best-epoch checkpoint already saved.')
    if cp.exists():
        ec=stream([sys.executable,'-u','-m','app.arcade_ml.audit_validation','--checkpoint',str(cp),'--device','cuda'])
        run_results.append({'id':x['id'],'status':'evaluated' if ec==0 else 'evaluation_error','checkpoint':str(cp)})
    else: run_results.append({'id':x['id'],'status':'no_checkpoint'})
    Path('/kaggle/working/arcade_run_progress.json').write_text(json.dumps(run_results,indent=2))
print(json.dumps(run_results,indent=2))

## Compare metrics and package results

Use segmentation metrics, candidate precision/recall/F1 at both IoU thresholds, target-size groups, component-area sweep, inference time, and checkpoint size together. All architectures and thresholds use validation labels, so the final comparison is validation-selected and may be optimistic. The generated visual folders contain 50 distinct examples each with image, mask, overlay, candidate boxes, probability map, and per-image metrics.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
report_root=PROJECT_ROOT/'reports/model_validation'
spec_by_id={x['id']:x for x in EXPERIMENTS}
comparison_ids=['best_stenosis_unet_optimized']+[x['id'] for x in EXPERIMENTS]
rows=[]
for run_id in comparison_ids:
    p=report_root/f"validation_metrics_{run_id}.json"
    if not p.is_file(): continue
    d=json.loads(p.read_text()); macro=d['selected_threshold_metrics']['per_image_macro']; glob=d['selected_threshold_metrics']['global']; cand=d['candidate_detection']['by_iou_threshold']; groups=d['small_target_performance']['groups']
    rows.append({'id':run_id,'architecture':d['checkpoint_metadata']['architecture'],'input':d['checkpoint_metadata']['image_size'],'threshold':d['selected_threshold'],'macro_dice':macro['dice'],'macro_iou':macro['iou'],'global_dice':glob['dice'],'precision':glob['precision'],'recall':glob['recall_sensitivity'],'f1':glob['f1'],'candidate_f1_iou30':cand['0.30']['f1'],'candidate_f1_iou50':cand['0.50']['f1'],'small_dice':groups['small']['macro_dice'],'medium_dice':groups['medium']['macro_dice'],'large_dice':groups['large']['macro_dice'],'seconds_per_image':d['inference_performance']['seconds_per_image'],'checkpoint_bytes':d['inference_performance']['checkpoint_size_bytes']})
comparison=pd.DataFrame(rows).sort_values('macro_dice',ascending=False)
comparison.to_csv(report_root/'kaggle_experiment_comparison.csv',index=False)
display(comparison)
if len(comparison):
    ax=comparison.set_index('id')[['macro_dice','macro_iou','candidate_f1_iou30','small_dice']].plot.bar(figsize=(15,6))
    ax.set_ylabel('Validation score'); ax.grid(axis='y',alpha=.25); plt.xticks(rotation=35,ha='right'); plt.tight_layout()
    plt.savefig(report_root/'kaggle_model_comparison.png',dpi=160); plt.show()
EXPORT=Path('/kaggle/working/arcade_model_export'); EXPORT.mkdir(exist_ok=True)
base_dir=EXPORT/'baseline_reference'; base_dir.mkdir(exist_ok=True)
for f in [report_root/'validation_metrics_best_stenosis_unet_optimized.json',report_root/'validation_per_image_best_stenosis_unet_optimized.csv',report_root/'validation_comparisons_best_stenosis_unet_optimized']:
    if f.is_file(): shutil.copy2(f,base_dir/f.name)
    elif f.is_dir(): shutil.copytree(f,base_dir/f.name,dirs_exist_ok=True)
for row in run_results:
    if row['status']!='evaluated': continue
    cp=Path(row['checkpoint']); d=EXPORT/cp.stem; d.mkdir(exist_ok=True)
    for f in [cp,cp.with_suffix(cp.suffix+'.threshold.json'),report_root/f'validation_metrics_{cp.stem}.json',report_root/f'validation_per_image_{cp.stem}.csv']:
        if f.is_file(): shutil.copy2(f,d/f.name)
    for f in [report_root/f'validation_comparisons_{cp.stem}',report_root/cp.stem]:
        if f.is_dir(): shutil.copytree(f,d/f.name,dirs_exist_ok=True)
if len(comparison): shutil.copy2(report_root/'kaggle_experiment_comparison.csv',EXPORT/'kaggle_experiment_comparison.csv')
shutil.make_archive('/kaggle/working/arcade_model_export','zip',EXPORT)
print('Download /kaggle/working/arcade_model_export.zip from the Kaggle notebook Output panel.')

## Transfer only a verified winner back to the app

Download `arcade_model_export.zip`; choose a model only after reviewing the complete report. Copy its `.pth` and matching threshold sidecar to **new unique filenames** under `backend\checkpoints` and retain the existing checkpoint. For SMP architectures, install `backend\requirements-ml.txt` in the backend virtual environment. From `backend`, validate the transferred checkpoint with `python -m app.arcade_ml.audit_validation --checkpoint checkpoints\<winner>.pth --device cpu`. After local inference, API, permissions, dashboard, and exports are verified, set `MODEL_CHECKPOINT` in the existing `backend\.env`. Never replace the old checkpoint during this procedure.

The notebook does not compute percentage narrowing or severity; the dataset lacks validated lumen boundaries, diameter measurements, and physical calibration.